In [1]:
bucket_name ='ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

In [2]:
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [3]:
role = sagemaker.get_execution_role()
#bucket= 'ibk-discovery-comercial-us-east-1-654654352211-artifacts'

#input_data_path = 'discovery/comercial/nsantilli/opc/principalidad/input/data_pr_v3_principales/' 
#'discovery/comercial/nsantilli/opc/principalidad/input/data_pr_v3/'
#output_data_path = 's3://ibk-discovery-comercial-us-east-1-654654352211-artifacts/discovery/comercial/nsantilli/opc/principalidad/output_V3_principales'
#path_prep_func = "discovery/comercial/nsantilli/utils/preprocessing_functions.py"

In [4]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu',
#'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.3.0-cpu-py311-ubuntu20.04-sagemaker'
#'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.3.0-cpu-py311-ubuntu20.04-sagemaker' #f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

team = 'RETAIL' # TODO 1: Colocar nombre del equipo: RETAIL, RIESGOS
name_ds = 'Hernandez Santiago' # TODO 2: Colocar mis apellidos y nombres
account = boto3.client('sts').get_caller_identity()['Account']

In [5]:
tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

In [6]:
#model = 'digclte' # TODO 7: Colocar nombre de la rama del modelo
#partition = '202411' # TODO 8: Colocar periodo de los datos

#arguments = [
#    '--model', model,
#    '--table-score', f'scr_{model}',
#    '--partition', partition,
#]

In [7]:
sklearn_processor = SKLearnProcessor(framework_version='0.20.0',
                                     base_job_name= 'Digitalizacion',
                                     instance_type='ml.r5.12xlarge',
                                     role=role,
                                     tags=tags,
                                     instance_count=1,volume_size_in_gb=30)

In [8]:
#sklearn_processor = SKLearnProcessor(
#    framework_version="1.2-1",
##    role=role,
 #   instance_type="ml.m5.4xlarge",
 #   instance_count=1,
 #   tags=tags)

In [9]:
#s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/

In [10]:
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

In [11]:
### Run process<
sklearn_processor.run(code = 'preprocessing_1.py',
                      
                      inputs = [ProcessingInput(source = 's3://{}/{}/DATA_INFERENCIA/'.format(bucket_name,model_prefix),
                                                destination = '/opt/ml/processing/input')],

                      
                      outputs = [
                                 ProcessingOutput(output_name='headers',
                                                  source = '/opt/ml/processing/headers',
                                                  destination = 's3://{}/{}/data_dev_model/'.format(bucket_name,model_prefix)),
                                 ProcessingOutput(output_name='train',
                                                  source = '/opt/ml/processing/train',
                                                  destination = 's3://{}/{}/data_dev_model/'.format(bucket_name,model_prefix)),
                                 
                                 ProcessingOutput(output_name='val',
                                                  source = '/opt/ml/processing/val',
                                                  destination = 's3://{}/{}/data_dev_model/'.format(bucket_name,model_prefix)),
                      
                                 ProcessingOutput(output_name='test',
                                                  source = '/opt/ml/processing/test',
                                                  destination = 's3://{}/{}/data_dev_model/'.format(bucket_name,model_prefix)),
                                ProcessingOutput(output_name='DIR_OBJ',
                                                  source = '/opt/ml/processing/DIR_OBJ',
                                              destination = 's3://{}/{}/MODEL/preproc_obj/'.format(bucket_name,model_prefix)),
                                ProcessingOutput(output_name='analisis',
                                                  source = '/opt/ml/processing/analisis',
                                                  destination = 's3://{}/{}/MODEL/df_analisis/'.format(bucket_name,model_prefix))
                                                                  
                                ]                      
                     )
preprocessing_job_description = sklearn_processor.jobs[-1].describe()

INFO:sagemaker:Creating processing-job with name Digitalizacion-2026-06-02-20-06-21-528


................./miniconda3/lib/python3.7/site-packages/sklearn/externals/joblib/externals/cloudpickle/cloudpickle.py:47: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  import imp
/miniconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LARGE_SPARSE_SUPPORTED = LooseVersion(scipy_version) >= '0.14.0'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 30.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 0.25.3
    Uninstalling pandas-0.25.3:
      Successfully uninstalled pandas-0.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker-sklearn-container 1.0 requires pandas==0.25.*, but you have pandas 1.1.5 which is incomp

In [12]:
df.head()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 df.head()                                                                                    │
│   2                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'df' is not defined

In [ ]:
df.head()